# 3.2 — Абляции, метрики по группам и out-of-distribution тесты

**Папка 3, подноутбук 2.** Анализ устойчивости и вклада компонентов: метрики по типам
грунта и режимам нагружения, абляции DPI-Flow и EVT-NeuralSSM, OOD-тесты (экстраполяция
«короткий → длинный горизонт» и удержанный регион слабых грунтов). Все рисунки и таблицы
— на английском.

## Окружение, данные и модели

In [2]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Найти корень репозитория по наличию pyproject.toml вверх по дереву."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display

from liquefaction_ai.viz import register_theme

register_theme()

# Если True — все фигуры сохраняются в results/figs (.html и .png)
SAVE_FIGS = True
DATA_DIR = REPO_ROOT / "data" / "demo_run"
MODELS_DIR = REPO_ROOT / "models"

import torch

from liquefaction_ai import load_population_artifact, prepare_benchmark_dataset
from liquefaction_ai.training import load_model_metadata, load_weights_into
from liquefaction_ai.models import (DPIFlow, EVTNeuralSSM, GRUBaseline, LSTMBaseline, RiskMLP, TCNBaseline, TransformerBaseline, FTTransformer,
                                    PINNBaseline, DeepStateBaseline, RealNVPFlow, NeuralSplineFlow,
                                    NeuralODENoPhysics, FlowNoODE, DPIEvtNet)
from liquefaction_ai.evaluation import collect_outputs, compute_metrics, english_metric_table

CLASS_REGISTRY = {"RiskMLP": RiskMLP, "GRUBaseline": GRUBaseline, "TCNBaseline": TCNBaseline, "LSTMBaseline": LSTMBaseline, "TransformerBaseline": TransformerBaseline, "FTTransformer": FTTransformer, "PINNBaseline": PINNBaseline, "DeepStateBaseline": DeepStateBaseline, "RealNVPFlow": RealNVPFlow, "NeuralSplineFlow": NeuralSplineFlow,
                  "DPIFlow": DPIFlow, "EVTNeuralSSM": EVTNeuralSSM, "DPIEvtNet": DPIEvtNet}
MODEL_NAMES = ["mlp_risk", "gru", "tcn", "lstm", "transformer", "ft_transformer", "pinn", "deepstate", "realnvp", "nsf", "dpi_flow", "evt_ssm", "dpi_evt"]

population, config = load_population_artifact(DATA_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
benchmark = prepare_benchmark_dataset(population, config, device)
test = benchmark["test"]


def load_trained(name):
    """Восстановить модель по сохранённым гиперпараметрам и весам."""
    hp, hist = load_model_metadata(MODELS_DIR, name)
    model = CLASS_REGISTRY[hp["model_type"]](**hp["model_kwargs"])
    load_weights_into(model, MODELS_DIR, name, device)
    return model, hp, hist
from liquefaction_ai.evaluation import (filter_split, grouped_metrics, is_holdout_region,
                                        run_quick_experiment, subsample_split)
from liquefaction_ai.constants import LOAD_DISPLAY_NAMES_EN, SOIL_DISPLAY_NAMES_EN
from liquefaction_ai.viz import bar, grouped_bar

## Metrics by soil type and loading regime (DPI-Flow)

In [3]:
dpi, hp, _ = load_trained("dpi_flow")
out = collect_outputs(dpi, test, config, device)
_, sample_df = compute_metrics("DPI-Flow", out, test, config)
sample_df["soil_en"] = sample_df["soil_type"].map(SOIL_DISPLAY_NAMES_EN)
sample_df["load_en"] = sample_df["load_mode"].map(LOAD_DISPLAY_NAMES_EN)
soil_metrics = grouped_metrics(sample_df, "soil_en")
load_metrics = grouped_metrics(sample_df, "load_en")
display(english_metric_table(soil_metrics).round(4))
grouped_bar(soil_metrics["soil_en"].tolist(),
            {"AUROC": soil_metrics["AUROC"].tolist(), "Trajectory RMSE": soil_metrics["mean_traj_rmse"].tolist()},
            title="DPI-Flow quality by soil type", ylabel="Value (–)",
            save=SAVE_FIGS, fig_id="3_2_metrics_by_soil").show()
grouped_bar(load_metrics["load_en"].tolist(),
            {"AUROC": load_metrics["AUROC"].tolist(), "Trajectory RMSE": load_metrics["mean_traj_rmse"].tolist()},
            title="DPI-Flow quality by loading regime", ylabel="Value (–)",
            save=SAVE_FIGS, fig_id="3_2_metrics_by_load").show()

,soil_en,N samples,Liquefaction rate,Mean predicted risk,Mean |ΔN_liq| (cycles),Mean log-error N_liq,Mean trajectory RMSE,Mean interval width,Physics violations,AUROC
3,Loam,23,0.5217,0.5655,2182.967529,1.4906,0.1163,0.3474,0.0,1.0
0,Clay,17,0.4706,0.4886,2775.836914,1.4454,0.0721,0.3286,0.0,1.0
2,Fine sand,17,0.4706,0.5212,2550.506592,1.9863,0.0963,0.3501,0.0,1.0
4,Medium sand,16,0.6875,0.7234,1517.492310,1.3689,0.1165,0.3608,0.0,1.0
1,Coarse sand,14,0.9286,0.9562,616.848389,0.8195,0.3031,0.4327,0.0,1.0
6,Silty sand,8,0.8750,0.8385,645.866394,1.0726,0.1250,0.3615,0.0,1.0
5,Sandy loam,6,0.5000,0.5397,2857.690186,1.6345,0.1192,0.3362,0.0,1.0


[save_figure] PNG для '3_2_metrics_by_soil' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[save_figure] PNG для '3_2_metrics_by_load' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Component ablations

In [4]:
sub = min(config.ablation_subset, 2400)
atrain = subsample_split(benchmark["train"], int(sub * 0.7), config.seed)
aval = subsample_split(benchmark["val"], int(sub * 0.15), config.seed + 1)
atest = subsample_split(benchmark["test"], int(sub * 0.15), config.seed + 2)
sd, pdim, qd = static_dim, prefix_dim, seq_dim = (benchmark["train"]["static"].shape[1],
    benchmark["train"]["prefix_summary"].shape[1], benchmark["train"]["seq_in"].shape[-1])
mcr, sl, pl = config.max_cycle_reference, config.seq_len, config.prefix_len
ablation_specs = [
    ("DPI-Flow (full)", DPIFlow(sd, pdim, sl, pl, mcr, calibration_steps=2, use_analytical_layer=True)),
    ("DPI-Flow w/o calibration", DPIFlow(sd, pdim, sl, pl, mcr, calibration_steps=0, use_analytical_layer=True)),
    ("DPI-Flow w/o probabilistic head", DPIFlow(sd, pdim, sl, pl, mcr, probabilistic=False, calibration_steps=2)),
    ("DPI-Flow w/o ODE layer", DPIFlow(sd, pdim, sl, pl, mcr, calibration_steps=0, use_analytical_layer=False)),
    # --- покомпонентные абляции (изоляция вклада ODE / flow / физики) ---
    ("ODE w/o flow", DPIFlow(sd, pdim, sl, pl, mcr, calibration_steps=2, use_analytical_layer=True, use_flow=False)),
    ("Flow w/o ODE", FlowNoODE(sd, pdim, sl)),
    ("Neural ODE w/o physics", NeuralODENoPhysics(sd, qd)),
    ("EVT (full)", EVTNeuralSSM(sd, pdim, qd, sl, pl, mcr)),
    ("EVT w/o trigger", EVTNeuralSSM(sd, pdim, qd, sl, pl, mcr, use_trigger_head=False)),
    ("EVT w/o post-event dynamics", EVTNeuralSSM(sd, pdim, qd, sl, pl, mcr, structured_post_event=False)),
    ("EVT w/o CRR damage", EVTNeuralSSM(sd, pdim, qd, sl, pl, mcr, use_crr_damage=False)),
]
ablation_rows = [run_quick_experiment(n, m, atrain, aval, atest, epochs=config.ablation_epochs,
                                      config=config, device=device) for n, m in ablation_specs]
ablation_df = pd.DataFrame(ablation_rows).sort_values("Traj_RMSE")
display(english_metric_table(ablation_df)[["Model", "Trajectory RMSE", "AUROC", "Brier"]].round(4))
bar(ablation_df["model"], ablation_df["Traj_RMSE"], title="Ablations: trajectory RMSE (lower is better)",
    ylabel="Trajectory RMSE (–)", color="#6610f2", horizontal=True,
    save=SAVE_FIGS, fig_id="3_2_ablations_rmse").show()

[DPI-Flow (full)] эпоха 01 | обучение=2.8125 | валидация=-0.2965


[DPI-Flow (full)] эпоха 02 | обучение=-0.5181 | валидация=-1.0636


[DPI-Flow w/o calibration] эпоха 01 | обучение=3.6790 | валидация=0.5227


[DPI-Flow w/o calibration] эпоха 02 | обучение=-0.1447 | валидация=-0.8691


[DPI-Flow w/o probabilistic head] эпоха 01 | обучение=2.4346 | валидация=-0.3743


...

[EVT w/o post-event dynamics] эпоха 02 | обучение=1.6031 | валидация=1.2655


[EVT w/o CRR damage] эпоха 01 | обучение=3.0217 | валидация=0.5518


[EVT w/o CRR damage] эпоха 02 | обучение=-0.1679 | валидация=-0.7734


,Model,Trajectory RMSE,AUROC,Brier
0,DPI-Flow (full),0.1417,0.9950,0.0738
4,ODE w/o flow,0.1432,1.0000,0.0340
2,DPI-Flow w/o probabilistic head,0.1442,0.9913,0.0648
7,EVT (full),0.1634,0.9975,0.0887
1,DPI-Flow w/o calibration,0.1647,0.9971,0.0561
3,DPI-Flow w/o ODE layer,0.1684,1.0000,0.0275
10,EVT w/o CRR damage,0.1770,0.9888,0.0905
6,Neural ODE w/o physics,0.2336,0.9938,0.1296
5,Flow w/o ODE,0.2582,1.0000,0.0359
9,EVT w/o post-event dynamics,0.3428,0.9905,0.0803


[save_figure] PNG для '3_2_ablations_rmse' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Out-of-distribution tests

In [5]:
train_meta = benchmark["train"]["meta"]; val_meta = benchmark["val"]["meta"]; test_meta = benchmark["test"]["meta"]
short_thr = float(train_meta["N_max"].quantile(0.60)); long_thr = float(test_meta["N_max"].quantile(0.80))
short_train = filter_split(benchmark["train"], train_meta["N_max"].to_numpy() <= short_thr)
short_val = filter_split(benchmark["val"], val_meta["N_max"].to_numpy() <= short_thr)
long_test = filter_split(benchmark["test"], test_meta["N_max"].to_numpy() >= long_thr)
e_thr = float(benchmark["meta"]["e"].quantile(0.75)); vs_thr = float(benchmark["meta"]["V_s"].quantile(0.25))
region_test = filter_split(benchmark["test"], is_holdout_region(test_meta, e_thr, vs_thr))
hold_train = filter_split(benchmark["train"], ~is_holdout_region(train_meta, e_thr, vs_thr))
hold_val = filter_split(benchmark["val"], ~is_holdout_region(val_meta, e_thr, vs_thr))
short_train = subsample_split(short_train, 1100, config.seed); short_val = subsample_split(short_val, 300, config.seed + 1)
long_test = subsample_split(long_test, 700, config.seed + 2)
hold_train = subsample_split(hold_train, 1100, config.seed + 3); hold_val = subsample_split(hold_val, 300, config.seed + 4)
region_test = subsample_split(region_test, 700, config.seed + 5)

def fresh(kind):
    if kind == "tcn": return TCNBaseline(sd, qd, 96)
    if kind == "dpi": return DPIFlow(sd, pdim, sl, pl, mcr, calibration_steps=2)
    return EVTNeuralSSM(sd, pdim, qd, sl, pl, mcr)

ood_rows = []
for disp, kind in [("TCN", "tcn"), ("DPI-Flow", "dpi"), ("EVT-NeuralSSM", "evt")]:
    ood_rows.append({**run_quick_experiment(disp, fresh(kind), short_train, short_val, long_test,
                     epochs=config.ablation_epochs, config=config, device=device), "test": "short→long"})
    ood_rows.append({**run_quick_experiment(disp, fresh(kind), hold_train, hold_val, region_test,
                     epochs=config.ablation_epochs, config=config, device=device), "test": "held-out region"})
ood_df = pd.DataFrame(ood_rows)
display(english_metric_table(ood_df)[["Model", "test", "Trajectory RMSE", "AUROC"]].round(4))
sl_df = ood_df[ood_df["test"] == "short→long"]; rg_df = ood_df[ood_df["test"] == "held-out region"]
grouped_bar(sl_df["model"].tolist(),
            {"short → long horizon": sl_df["Traj_RMSE"].tolist(), "held-out region": rg_df["Traj_RMSE"].tolist()},
            title="Out-of-distribution trajectory RMSE (lower is better)", ylabel="Trajectory RMSE (–)",
            save=SAVE_FIGS, fig_id="3_2_ood_rmse").show()

[TCN] эпоха 01 | обучение=0.3286 | валидация=0.2718


[TCN] эпоха 02 | обучение=0.2433 | валидация=0.0447


[TCN] эпоха 01 | обучение=0.3054 | валидация=0.2380


[TCN] эпоха 02 | обучение=0.1972 | валидация=-0.0974


[DPI-Flow] эпоха 01 | обучение=4.3042 | валидация=-0.3823


...

[EVT-NeuralSSM] эпоха 02 | обучение=1.6313 | валидация=1.3255


[EVT-NeuralSSM] эпоха 01 | обучение=3.6593 | валидация=2.6650


[EVT-NeuralSSM] эпоха 02 | обучение=0.9545 | валидация=-0.5077


,Model,test,Trajectory RMSE,AUROC
0,TCN,short→long,0.2152,0.9231
1,TCN,held-out region,0.3561,1.0000
2,DPI-Flow,short→long,0.1820,1.0000
3,DPI-Flow,held-out region,0.2014,1.0000
4,EVT-NeuralSSM,short→long,0.4769,1.0000
5,EVT-NeuralSSM,held-out region,0.2321,1.0000


[save_figure] PNG для '3_2_ood_rmse' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Итог

Абляции подтверждают вклад компонентов; структурированные модели устойчивее в OOD.
Дальше — **3.3 разбор кейсов**.

In [6]:
# === OOD-разбиение: качество по площадкам, типам грунтов и диапазонам CSR ===
# Для основной модели (DPI-Flow) считаем метрики по группам — генерализация по доменам.
from liquefaction_ai.evaluation import grouped_metrics
dpi_full, _, _ = load_trained("dpi_flow")
_out = collect_outputs(dpi_full, test, config, device)
_, sdf = compute_metrics("DPI-Flow", _out, test, config)
sdf["soil_en"] = sdf["soil_type"].map(SOIL_DISPLAY_NAMES_EN)
import numpy as np
sdf["CSR_bin"] = pd.cut(sdf["CSR_base"], [0, 0.12, 0.18, 0.25, 1.0],
                         labels=["CSR≤0.12", "0.12–0.18", "0.18–0.25", "CSR>0.25"])
_show = ["samples", "mean_traj_rmse", "mean_nliq_log_err", "physics_violation_rate", "AUROC"]
for dim, label in [("object", "площадка (site)"), ("soil_en", "тип грунта"), ("CSR_bin", "диапазон CSR")]:
    g = grouped_metrics(sdf, dim)
    print(f"\n=== OOD по: {label} ===")
    display(english_metric_table(g[[dim] + _show]).round(4))
    bar(g[dim].astype(str), g["mean_traj_rmse"], title=f"DPI-Flow: PPR RMSE by {dim}",
        ylabel="Trajectory RMSE (–)", color="#0b6efd", horizontal=True,
        save=SAVE_FIGS, fig_id=f"3_2_ood_{dim}").show()


=== OOD по: площадка (site) ===


,object,N samples,Mean trajectory RMSE,Mean log-error N_liq,Physics violations,AUROC
10,Штормовое разжижение/856-23 3-й Хорошевский ...,21,0.1532,1.3783,0.0,0.963
2,Потенциал разжижения/852-23 3-й Хорошевский ...,17,0.1488,0.8338,0.0,NaN
0,Потенциал разжижения/333-24 Дубнинская - Plaxi...,13,0.1044,1.0812,0.0,NaN
1,Потенциал разжижения/338-24 АД Москва - Казань...,12,0.1186,1.2618,0.0,NaN
3,Штормовое разжижение/196-24 ул. Б. Татарская,10,0.1321,1.5948,0.0,1.000
9,Штормовое разжижение/818-23 Руза гора - plaxis...,8,0.0756,2.3199,0.0,1.000
5,Штормовое разжижение/255-24 Б. Садовая 8 - pla...,6,0.0964,2.8872,0.0,NaN
6,Штормовое разжижение/390-24 Шаболовская -Plaxi...,6,0.0656,1.5857,0.0,NaN
8,Штормовое разжижение/638-24 Ленинградское шоссе,5,0.3329,0.7139,0.0,1.000
7,Штормовое разжижение/613-23 Тушино - plaxis (23),2,0.0839,2.3045,0.0,NaN


[save_figure] PNG для '3_2_ood_object' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido




=== OOD по: тип грунта ===


,soil_en,N samples,Mean trajectory RMSE,Mean log-error N_liq,Physics violations,AUROC
3,Loam,23,0.1163,1.4906,0.0,1.0
0,Clay,17,0.0721,1.4454,0.0,1.0
2,Fine sand,17,0.0963,1.9863,0.0,1.0
4,Medium sand,16,0.1165,1.3689,0.0,1.0
1,Coarse sand,14,0.3031,0.8195,0.0,1.0
6,Silty sand,8,0.1250,1.0726,0.0,1.0
5,Sandy loam,6,0.1192,1.6345,0.0,1.0


[save_figure] PNG для '3_2_ood_soil_en' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido




=== OOD по: диапазон CSR ===


/sessions/determined-cool-fermat/mnt/liquefaction-ai/src/liquefaction_ai/evaluation/metrics.py:566: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sample_df.groupby(group_col)


,CSR_bin,N samples,Mean trajectory RMSE,Mean log-error N_liq,Physics violations,AUROC
1,0.12–0.18,39,0.0876,1.4586,0.0,0.9857
0,CSR≤0.12,37,0.0831,2.0282,0.0,1.0000
3,CSR>0.25,14,0.4235,0.3689,0.0,NaN
2,0.18–0.25,11,0.0853,0.6628,0.0,NaN


[save_figure] PNG для '3_2_ood_CSR_bin' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

